In [30]:
import sys 
sys.path.append( "../")
sys.path.append( "../../")

from datetime import datetime 

from typing import Dict, List, Tuple, Optional, Any 
from pathlib import Path 
from load_semantics import load_semantics, load_idiom_rules 
from semantics.semantic_models import * 
 
from typing import Any, Dict
import json
import yaml
from get_llm_model import *
import pandas as pd, numpy as np
import duckdb
import pprint 



In [31]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year
display( inj.sample(8))


llm = azure_llm_if()
print( llm )

,DATE,NAME,WATER_INJECTION_VOLUME,SECTOR,ZONE,SUBZONE,WELL_TYPE,DAY,MONTH,YEAR
77,2022-05-02,I1,1573.000,1,WARA,WARA1,Injector,2,5,2022
409,2017-05-02,I5,1208.000,1,WARA,WARA1,Injector,2,5,2017
154,2020-08-02,I2,449.000,1,WARA,WARA1,Injector,2,8,2020
420,2018-04-02,I5,767.194,1,WARA,WARA1,Injector,2,4,2018
276,2022-08-02,I3,1067.230,1,WARA,WARA1,Injector,2,8,2022
392,2015-12-02,I5,0.000,1,WARA,WARA1,Injector,2,12,2015
386,2023-08-02,I4,1184.740,1,WARA,WARA1,Injector,2,8,2023
129,2018-07-02,I2,1584.000,1,WARA,WARA1,Injector,2,7,2018


client=<openai.resources.chat.completions.completions.Completions object at 0x7cc06d391f30> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7cc06d3905e0> root_client=<openai.lib.azure.AzureOpenAI object at 0x7cc06d3903d0> root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x7cc06d3902e0> model_name='gpt-4o' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True azure_endpoint='https://openai-if-test.openai.azure.com/' deployment_name='gpt-4' openai_api_version='2024-12-01-preview' openai_api_type='azure'


In [32]:
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
 
#pprint.pprint(semantic_catalog.tables[0].model_dump_json() )

known_table_models = { item.name: item for item in semantic_catalog.tables }


In [36]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year

pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year


df_dict = {'injectors': inj, 'producers':pinj }

In [39]:

from typing import Iterable, Union


class Catalog:

    def __init__(self):
        self.tables: Dict[str, TableCard] = {}

    def clear(self):
        self.tables = {} 

    def initialize_from_named_dataframes( self, df_dict: Dict[str,pd.DataFrame], named_table_models ):
        
        self.clear() 

        for name,df in df_dict.items():
            model = named_table_models.get(name, None)
            if model:
                model.row_count = df.shape[0]

                dt = datetime.now() if hasattr(datetime, "now") else datetime.datetime.now()  
                model.creation_date = str( dt )
                
                self.tables[name] = model
            else:
                raise ValueError(f"Table named {name} is not in the known tables catalog")
       
    def register_table(self, table_card: TableCard ):
        dt = datetime.now() if hasattr(datetime, "now") else datetime.datetime.now()  
             
        table_card.creation_date = str( dt )
        self.tables[ table_card.name ] = table_card

    @staticmethod  
    def dataframe_to_table_card( df: pd.DataFrame, name, description, kind:Literal['base','derived'], **kwargs):
        dt = datetime.now() if hasattr(datetime, "now") else datetime.datetime.now()  
             
        cols = [ ColumnCard( name = col, data_type = str(df[col].dtype), description = None) for col in df.columns] 
        table_card = TableCard(
            name=name,
            description=description,
            kind = kind, 
            creation_date=str( dt ),
            row_count=df.shape[0],
            columns = cols,
            relationships = [] if not kwargs else kwargs.get('relationships', []),
            sql_examples  = [] if not kwargs else kwargs.get('sql_exampled',  [])

        )

        return table_card

    def __repr__(self) -> str:
        return self.snapshot()

    def snapshot(self, input_tables: None | TableCard | Iterable[TableCard] = None ) -> str: # pyright: ignore[reportArgumentType]
        
        table_blocks: list[str] = []
        
        items = (
            self.tables.values()
            if input_tables is None
            else [input_tables]
            if isinstance(input_tables, TableCard)
            else input_tables if isinstance(input_tables,Iterable)
            else list(input_tables)
        )
        '''
        s = "tables:\n"
        for tc in  sorted( items,  key=lambda x: x.name):
            s1 = f" - table:{tc.name}\n{tc.description}"
            cols = ""
            s = s + s1 

        return s 
        '''
 

        for tc in  sorted( items,  key=lambda x: x.name):
            d = tc.model_dump(
                exclude_none=True,
                exclude_defaults=True,
                exclude_unset = True,
                mode = 'json',
                #exclude = {'columns'}
            )

            block = yaml.dump(
                {"table": [d]},
                sort_keys=False,
                allow_unicode=True,
                width=1000  # avoid wrapping
            ).rstrip()

            table_blocks.append(block)
            table_blocks.append("")


        return "\n".join(table_blocks) 



    

    def __getitem__(self, value):
        
        cards = None 
        if isinstance(value, slice):
            cards =  list(self.tables.values())[value] 

        if isinstance(value, str ):
            cards = self.tables[value]

        if isinstance(value, Iterable ):
            cards = [ self.tables[v] for v in value] 
 

        return cards  

catalog = Catalog()
catalog.initialize_from_named_dataframes( df_dict, known_table_models)

df = pd.DataFrame( {'name':['a','b'], 'age':[1,2]})
catalog.register_table( catalog.dataframe_to_table_card(df, name='aaa',description='sddgf', kind='derived'))
print(catalog.snapshot)#( catalog['aaa', 'injectors' ] ))


<bound method Catalog.snapshot of table:
- name: aaa
  description: sddgf
  kind: derived
  creation_date: '2026-04-18 19:41:13.713330'
  row_count: 2
  columns:
  - name: name
    data_type: object
  - name: age
    data_type: int64

table:
- name: injectors
  description: Water-injection time series per injector well, subzone and sector
  kind: base
  creation_date: '2026-04-18 19:41:13.712454'
  row_count: 490
  columns:
  - name: DATE
    data_type: timestamp
    description: Timestamp of injection observation.
  - name: NAME
    data_type: string
    description: Injector well name. Unique identifier for the injector well
  - name: WATER_INJECTION_VOLUME
    data_type: float
    description: Injected water volume for the period.
  - name: SUBZONE
    data_type: string
    description: Subzone name. A subzone indicates vertical interval
  - name: SECTOR
    data_type: integer
    description: Sector identifier. A sector indicates a geographical location
  - name: YEAR
    data_type

In [ ]:
from langchain_core.tools import StructuredTool, Tool

class SmartData:

    def __init__(self):
        self._catalog = Catalog() 
        self.conn= duckdb.connect()


    def execute_sql( self, sql:str):#, table_description:str ):
        """
        materialize a table by executing sql.  
        Args:
            sql(str): sql quiery to execute. Must start with WITH or SELECT 
            table_description: brief description of the resulting table  
        """
        print('materializing')
        print('sql', sql)
        print('description', table_description)
        result = self.conn.execute(sql).fetchdf()
        return result 

    def clear( self ):
        self._catalog.clear()
        self.conn.close()
        self.conn= duckdb.connect()

    def initialize_from_named_dataframes( self, df_dict: Dict[str,pd.DataFrame], named_table_models ):
        self.clear()

        try:
            conn = self.conn
            self._catalog.initialize_from_named_dataframes( df_dict, named_table_models )
            
            for name, df in df_dict.items():
                df = self.sanitize_df(df)
                conn.register(name, df)

        except Exception as e:
            print( str(e))
            self.clear()
            
    def catalog_snapshot(self, input_tables: None | str | Iterable[str] = None) -> str:
        """
        Returns schema and description of all tables (base and derived) in the database
        """
        if input_tables is None:
            return self._catalog.snapshot()
        if isinstance(input_tables, str):
            card = self._catalog.tables[input_tables]
            return self._catalog.snapshot(card)
        if isinstance(input_tables, Iterable):
            cards = [self._catalog.tables[name] for name in input_tables]
            return self._catalog.snapshot(cards)
        raise TypeError(f"{type(input_tables).__name__} is not supported")
    
    def register_derived_table(self, df:pd.DataFrame, name:str, table_description:str ):
        card = Catalog.dataframe_to_table_card( df, name, table_description, 'derived' )
        self._catalog.register_table( card )
        self.conn.register(name, df)

    def sanitize_df(self, df):

        df = df.copy()
        return df 
    
        # Ensure index is not problematic
        if df.index.name is not None or not isinstance(df.index, pd.RangeIndex):
            df = df.reset_index()

        # Attempt to convert object columns
        for col in df.columns:
            if df[col].dtype == "object":
                # try datetime
                converted = pd.to_datetime(df[col], errors="ignore")
                if not pd.api.types.is_object_dtype(converted):
                    df[col] = converted
                    continue

                # try numeric
                converted = pd.to_numeric(df[col], errors="ignore")
                if not pd.api.types.is_object_dtype(converted):
                    df[col] = converted

        return df

    def get_table_names( self ):
        """Returns the table names"""
        return [name for name in self._catalog.tables ] 
    
    def get_tables_creation_datetime( self )-> Dict[str,str]  :
        """Returns the creation date of each table"""
        return { t: v.creation_date  for t,v in self._catalog.tables.items() }   # pyright: ignore[reportReturnType]
    
    def get_tables_brief_description( self ):
        """Returns a brief textual description of the tables"""
        return { t: v.description  for t,v in self._catalog.tables.items() }  



data = SmartData()
data.initialize_from_named_dataframes( df_dict, known_table_models)
data.register_derived_table( df, "a derived table", 'Something created on the fly')

print(data.catalog_snapshot())# ['injectors'] ))



table:
- name: a derived table
  description: Something created on the fly
  kind: derived
  creation_date: '2026-04-18 20:02:52.386414'
  row_count: 2
  columns:
  - name: name
    data_type: object
  - name: age
    data_type: int64

table:
- name: injectors
  description: Water-injection time series per injector well, subzone and sector
  kind: base
  creation_date: '2026-04-18 20:02:52.363585'
  row_count: 490
  columns:
  - name: DATE
    data_type: timestamp
    description: Timestamp of injection observation.
  - name: NAME
    data_type: string
    description: Injector well name. Unique identifier for the injector well
  - name: WATER_INJECTION_VOLUME
    data_type: float
    description: Injected water volume for the period.
  - name: SUBZONE
    data_type: string
    description: Subzone name. A subzone indicates vertical interval
  - name: SECTOR
    data_type: integer
    description: Sector identifier. A sector indicates a geographical location
  - name: YEAR
    data_typ

In [ ]:

class SmartDataTools:

    def __init__(self, data:SmartData):
        self._data = data 

    # ----------------------------------------
    def emit_step_by_step_plan(self, step_by_step_plan: str) -> str:
        """
        Record and display the execution plan generated by the agent.
        """
        # return the plan so it shows in tool output.
        return step_by_step_plan

    def get_table_names( self ):
        """Returns the table names"""
        return  self._data.get_table_names() 
    
    def get_tables_creation_datetime( self )-> Dict[str,str]  :
        """Returns the creation date of each table"""
        return self._data.get_tables_creation_datetime()
        
    def get_tables_brief_description( self ):
        """Returns a brief textual description of the tables"""
        return self._data.get_tables_brief_description() 

    def get_tools(self):
        tools = []
        for name in dir(self):
            if name.startswith("_") or name == "get_tools":
                continue
                
            attr = getattr(self, name)
            if not attr.__doc__:
                continue 


            if callable(attr) and attr.__doc__:
                tools.append(
                    StructuredTool.from_function(
                        func=attr,
                        name=name,
                        description=attr.__doc__,
                    )
                )
        return tools

    def materialize_sql( self, sql:str, materialized_table_name:str, table_description:str ):
        """
        materialize a table by executing sql.  
        Args:
            sql(str): sql quiery to execute. Must start with WITH or SELECT 
            materialized_table_name: name given to the new table. 
            table_description: brief description of the resulting table  
        """
        print('materializing')
        print('sql', sql)
        print('description', table_description)

        retries = 0 

        try:
            df_result = self._data.execute_sql(sql)
            self._data.register_derived_table( df_result, materialized_table_name, table_description)
        
            return f"Observation: table {materialized_table_name} created "
        except Exception as e:

        
            return "SQL instruction failed" 



smart_data_tools = SmartDataTools( data=data)
smart_data_tools.get_tools()




[StructuredTool(name='emit_step_by_step_plan', description='Record and display the execution plan generated by the agent.', args_schema=<class 'langchain_core.utils.pydantic.emit_step_by_step_plan'>, func=<bound method SmartDataTools.emit_step_by_step_plan of <__main__.SmartDataTools object at 0x7cc06d3ba620>>),
 StructuredTool(name='get_table_names', description='Returns the table names', args_schema=<class 'langchain_core.utils.pydantic.get_table_names'>, func=<bound method SmartDataTools.get_table_names of <__main__.SmartDataTools object at 0x7cc06d3ba620>>),
 StructuredTool(name='get_tables_brief_description', description='Returns a brief textual description of the tables', args_schema=<class 'langchain_core.utils.pydantic.get_tables_brief_description'>, func=<bound method SmartDataTools.get_tables_brief_description of <__main__.SmartDataTools object at 0x7cc06d3ba620>>),
 StructuredTool(name='get_tables_creation_datetime', description='Returns the creation date of each table', arg

In [ ]:

def build_prompt( user_query:str):
    

In [43]:

def sanitize_df(df):

    df = df.copy()

    # Ensure index is not problematic
    if df.index.name is not None or not isinstance(df.index, pd.RangeIndex):
        df = df.reset_index()

    # Attempt to convert object columns
    for col in df.columns:
        if df[col].dtype == "object":
            # try datetime
            converted = pd.to_datetime(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted
                continue

            # try numeric
            converted = pd.to_numeric(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted

    return df

def register_table(conn, name: str, df, registry: set, columns: Dict[str, Any]):
    df = sanitize_df(df)
    conn.register(name, df)


con = duckdb.connect()
con.register("injectors", sanitize_df(inj))

/tmp/ipykernel_1224/2207352061.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_1224/2207352061.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_1224/2207352061.py:19: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_numeric(df[col], errors="ignore")
/tmp/ipykernel_1224/2207352061.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  converted = p

In [44]:

con = duckdb.connect()
con.register("injectors", sanitize_df(inj))


/tmp/ipykernel_1224/2207352061.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_1224/2207352061.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_1224/2207352061.py:19: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_numeric(df[col], errors="ignore")
/tmp/ipykernel_1224/2207352061.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  converted = p

In [9]:
from load_semantics import load_semantics, load_idiom_rules 
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
 

In [ ]:

query = 'fdfgdfgdg'
prompt_template = """
You are an expert SQL generator for {idiom} based on the 
following database schema and description:

# Tables:
{context_lines}   

# Rules:
- Generate ONLY the SQL instruction. 
- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

ALWAYS use {idiom} compliant SQL syntax.
Examples:
{idiom_examples}

 
# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.

"Current date" refers to the MAX("DATE") in the dataset.

Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

# Task: 
{user_query}

# {idiom} SQL:
"""


idiom_name = "duckdb"
idiom_examples = idiom_context# "\n".join([f"- {k}: {v}" for k, v in idioms[idiom_name].items()])

context_dict = semantic_catalog.tables[0].model_dump(exclude_none=True)
context_yaml = yaml.dump(context_dict, sort_keys=False)


# FIX: Changed 'idioms' to 'idiom' to match the template placeholder
prompt = prompt_template.format(
    user_query=query, 
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
) 

# response = llm.invoke(prompt)
print(prompt)


You are an expert SQL generator for duckdb based on the 
following database schema and description:

# Tables:
name: injectors
description: Water-injection time series per injector well, subzone and sector
kind: base
creation_date: '2026-04-18 19:45:28.336641'
row_count: 490
columns:
- name: DATE
  data_type: timestamp
  description: Timestamp of injection observation.
- name: NAME
  data_type: string
  description: Injector well name. Unique identifier for the injector well
- name: WATER_INJECTION_VOLUME
  data_type: float
  description: Injected water volume for the period.
- name: SUBZONE
  data_type: string
  description: Subzone name. A subzone indicates vertical interval
- name: SECTOR
  data_type: integer
  description: Sector identifier. A sector indicates a geographical location
- name: YEAR
  data_type: integer
  description: Year component of DATE.
- name: MONTH
  data_type: integer
  description: Month component of DATE.
- name: DAY
  data_type: integer
  description: Day 

In [47]:
query1 = "how many wells are there?"
query2 = "What is the total water injection volume by year?"
query3 = "Tell me the mean yearly water injection volume for each subzone"
query4 = "rank wells by their variability (std) in water injection volume (the higher the grater the rank)?"
query5 = "whats the frequency of observations in the dataset (D, M, Y) ?"
query6 = "summarize the injection data"
query7 = "Which well had the single highest WATER_INJECTION_VOLUME reading at any point in time and what was that reading?"
query8 = "What is the average monthly injection volume per well grouped by NAME and MONTH?"
query9 = """For each SUBZONE compute the year-over-year percentage change in total injection 
volume and report the largest drop
"""


queries = [
    #(query1, lambda x: int(x.loc[0,:].values[0]) == 5, 1),
    #(query2, lambda x: abs(float(x.loc[ x["YEAR"] == 2016, :].values[0][1]) - 56202.305) < 0.01, 2),
    #(query3, lambda x: abs(1306.96 - float(x.set_index(["YEAR", "SUBZONE"]).iloc[:, 0].loc[(2019, "WARA1")])) < 0.01, 2),
    #(query4, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    #(query7, lambda x: (x.shape[0] == 1 and (x.iloc[0]["NAME"] == "I1") and abs(float(x.iloc[0]["WATER_INJECTION_VOLUME"]) - 3537.0) < 0.1), 1),
    (query8, lambda x: False, 2),
    #(query9, lambda x: False, 3),
]

for n, query_item in enumerate(queries):
    query, checking_fn, complexity = query_item
    print(60 * "=")
    print(f"Complexity index: {complexity}")
    print(query)
   
    prompt = prompt_template.format(
        user_query=query, 
        context_lines=context_yaml, 
        idiom=idiom_name, 
        idiom_examples=idiom_examples
    )
    response = llm.invoke(prompt)

    print(response)

    # execute the generated SQL
    sql = response.content.strip()
    print(sql)

    try:
        result = con.execute(sql).fetchdf()
        display(result.sample(min(3, result.shape[0])))

        # check result
        print("success", checking_fn(result))
    except Exception as e:
        print("error", e)


Complexity index: 2
What is the average monthly injection volume per well grouped by NAME and MONTH?
content='SELECT NAME, MONTH, AVG(WATER_INJECTION_VOLUME) AS avg_monthly_injection_volume\nFROM injectors\nGROUP BY NAME, MONTH' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 740, 'total_tokens': 770, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DW5u5EVVpI8EPPAQ1hdlH2Wo22XGt', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered

,NAME,MONTH,avg_monthly_injection_volume
56,I5,4,993.540625
19,I4,12,782.573000
22,I4,4,791.206000


success False


In [28]:
from pprint import pprint 
pprint(response.usage_metadata)
pprint(response.response_metadata["token_usage"]) 

{'input_token_details': {'audio': 0, 'cache_read': 0},
 'input_tokens': 820,
 'output_token_details': {'audio': 0, 'reasoning': 0},
 'output_tokens': 237,
 'total_tokens': 1057}
{'completion_tokens': 237,
 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                               'audio_tokens': 0,
                               'reasoning_tokens': 0,
                               'rejected_prediction_tokens': 0},
 'prompt_tokens': 820,
 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
 'total_tokens': 1057}


# Now lets make it agentic and add some sort of structured output 

## Generate a prompt_builder that RAGs the questions, reasoning and sql 
## Add custom tools 


In [ ]:
response.response_metadata['token_usage']['prompt_tokens']

In [ ]:


class SmartData:
    def __init__(self, llm=None):
        self.con = duckdb.connect()
        self.tables = set()
        self.semantic = {}
        self.semantic_model = load_semantic_model()
        self.columns = {}
        self._llm = llm

    # -----------------------------
    # LLM PROPERTY
    # -----------------------------
    @property
    def llm(self):
        return self._llm

    @llm.setter
    def llm(self, model):
        self._llm = model

    # -----------------------------
    # CORE EXECUTION
    # -----------------------------
    def execute_query(self, sql: str):
        return self.con.execute(sql).fetchdf()

    # -----------------------------
    # TABLE REGISTRATION
    # -----------------------------
    def register_tables(self, tables: Dict[str, Any]):
        register_tables(self.con, tables, self.tables, self.columns)

    def register_table(self, name: str, df):
        register_table(self.con, name, df, self.tables, self.columns)

    # -----------------------------
    # SEMANTIC REGISTRATION
    # -----------------------------
    def register_semantic(self, name: str, description: str):
        if name not in self.tables:
            raise ValueError(f"Table '{name}' is not registered")
        self.semantic[name] = description

    # -----------------------------
    # SQL GENERATION
    # -----------------------------
    def generate_sql(self, user_query: str, **kwargs) -> str:
        if self._llm is None:
            raise ValueError("LLM is not set")

        context_lines = []
        for table in self.tables:
            desc = self.semantic.get(table, "")
            cols = self.columns.get(table, [])
            context_lines.append(
                f"Table: {table}\nDescription: {desc}\nColumns: {', '.join(cols)}"
            )

        context = "\n\n".join(context_lines)

        prompt = f"""
You are an expert SQL generator for DuckDB.

Available tables:
{context}

User request:
{user_query}

Generate a valid DuckDB SQL query only.
"""

        return self._llm(prompt, **kwargs)


llm = azure_llm_if()
print( llm )

